# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
print("Token added successfully:", hf_token is not None)
file_c = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token)
file_f = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token)
df_c = pd.read_parquet(file_c)
df_f = pd.read_parquet(file_f)
print(df_c.shape, df_f.shape)
df_c.head()

Token added successfully: True
(519606, 26) (9841378, 30)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [18]:
df_f.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit Of Analysis:** 1 row = 1 unique webpage (content_hash_id), aggregated over March 2026.
* **Time Window:** month=2026-03, loaded directly from the warehouse partition (verified below)

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
page_stats=df_f.groupby('content_hash_id').agg(
    total_clicks=('gsc_clicks','sum'),
    total_impressions=('gsc_impressions', 'sum'),
    avg_position=('gsc_sum_position', 'mean')
).reset_index()
page_stats.head()

,content_hash_id,total_clicks,total_impressions,avg_position
0,content_000005d4ced12088,0,86,199.967742
1,content_00001e488b74b799,0,0,0.000000
2,content_00007bd2985b77c3,0,47,8.032258
3,content_00008950670cb6b5,0,0,0.000000
4,content_0000a348850eb1fc,0,0,0.000000


In [ ]:
# Time Window Claim
df_f['report_date']=pd.to_datetime(df_f['report_date'])
max_date=df_f['report_date'].max()
min_date=df_f['report_date'].min()
print(f"The max date is {max_date} & the min date is {min_date}")

The max date is 2026-03-31 00:00:00 & the min date is 2026-03-01 00:00:00


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.

* **Features:** avg_position, total_clicks, total_impressions, click_trend, word_count, content_type all knowable by end of March 2026, all raw aggregates from fact_content_daily_performance, none derived from the label.
* **Label:** opportunity_flags count of true risk conditions (declining, page-one, stale stale defined as days_since_update >= 180, sourced from dim_content.content_updated_date), transparent, no invented weights.
* **Context:** content_hash_id (page identifier, not predictive).
* **Excluded:** raw URLs, client identifiers — anonymization/privacy. cpc, search_volume, word_count not present in the real warehouse tables, were starter-CSV-only simplifications.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
mid = df_f['report_date'].median()
page_stats = df_f.groupby('content_hash_id').agg(
    total_clicks=('gsc_clicks', 'sum'),
    total_impressions=('gsc_impressions', 'sum'),
    avg_position=('gsc_avg_position', 'mean')).reset_index()
early_clicks = df_f[df_f['report_date'] < mid].groupby('content_hash_id')['gsc_clicks'].sum()
late_clicks = df_f[df_f['report_date'] >= mid].groupby('content_hash_id')['gsc_clicks'].sum()
click_trend = (late_clicks - early_clicks).rename('click_trend')
df = page_stats.merge(click_trend, on='content_hash_id', how='left')
print(df.shape)
df.head()
df['is_declining'] = (df['click_trend'] < 0).astype(int)
df['is_page_one'] = (df['avg_position'] <= 10).astype(int)
df['opportunity_flags'] = df['is_declining'] + df['is_page_one']
features = ['avg_position', 'total_clicks', 'total_impressions', 'click_trend']
label = ['opportunity_flags']
context = ['content_hash_id']
print(f"Features ({len(features)}): {features}")
print(f"Label (1): {label}")
df_c['content_updated_date'] = pd.to_datetime(df_c['content_updated_date'])
reference_date = df_f['report_date'].max()
df_c['days_since_update'] = (reference_date - df_c['content_updated_date']).dt.days
df = df.merge(
    df_c[['content_hash_id', 'days_since_update', 'word_count', 'content_type']],
    on='content_hash_id',
    how='left')
df['is_stale'] = (df['days_since_update'] >= 180).astype(int)
df['opportunity_flags'] = df['is_declining'] + df['is_page_one'] + df['is_stale']
def reason_codes(row):
    reasons = []
    if row['is_declining']: reasons.append('declining traffic')
    if row['is_page_one']: reasons.append('page-one decay risk')
    if row['is_stale']: reasons.append('stale content')
    return ', '.join(reasons) if reasons else 'no major flags'
df['reason_code'] = df.apply(reason_codes, axis=1)
features = ['avg_position', 'total_clicks', 'total_impressions', 'click_trend', 'word_count']
label = ['opportunity_flags']
context = ['content_hash_id', 'content_type']
print(f"Features ({len(features)}): {features}")
print(f"Label (1): {label}")
print(f"Context ({len(context)}): {context}")
print(df[['content_hash_id', 'is_declining', 'is_page_one', 'is_stale', 'opportunity_flags', 'reason_code']].head())
print("\nFlag distribution:")
print(df['opportunity_flags'].value_counts().sort_index())

(331437, 5)
Features (4): ['avg_position', 'total_clicks', 'total_impressions', 'click_trend']
Label (1): ['opportunity_flags']
Features (5): ['avg_position', 'total_clicks', 'total_impressions', 'click_trend', 'word_count']
Label (1): ['opportunity_flags']
Context (2): ['content_hash_id', 'content_type']
            content_hash_id  is_declining  is_page_one  is_stale  \
0  content_000005d4ced12088             0            0         0   
1  content_00001e488b74b799             0            0         0   
2  content_00007bd2985b77c3             0            1         0   
3  content_00008950670cb6b5             0            0         0   
4  content_0000a348850eb1fc             0            0         0   

   opportunity_flags          reason_code  
0                  0       no major flags  
1                  0       no major flags  
2                  1  page-one decay risk  
3                  0       no major flags  
4                  0       no major flags  

Flag distribution:


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Time Window: This starter snapshot (content_refresh_anonymized.csv) has no date/month column — it's a single static cross-section, not time-partitioned. A real time window (e.g. month=2026-03) will apply once we move to the Hugging Face warehouse release in Week 3.

* **Grain Check:** Confirming zero duplicate content_id entries.
* **Row Count & Completeness:** Measuring overall row count and non-null availability across features.
* **Availability Filter (IS TRUE check):** Verifying how many rows survive when requiring valid, non-null feature values.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dup = df.duplicated(subset=['content_hash_id']).sum()
print(f"Duplicate grains: {dup}")
total_rows = len(df)
print(f"Total rows (pages): {total_rows:,}")
print("\nNull check on features:")
print(df[features].notnull().sum())
is_complete_row = df[features].notnull().all(axis=1) & df['content_hash_id'].notnull()
surviving_rows = is_complete_row.sum()
print(f"\nSurviving rows after availability filter: {surviving_rows:,} / {total_rows:,} ({surviving_rows/total_rows*100:.1f}%)")


Duplicate grains: 0
Total rows (pages): 331,437

Null check on features:
avg_position         176738
total_clicks         331437
total_impressions    331437
click_trend          319758
word_count           224008
dtype: int64

Surviving rows after availability filter: 118,953 / 331,437 (35.9%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What can this data never tell you?**

This data can't tell us why a page is declining (algorithm update vs. content quality vs. competitor changes)only that it's declining. It also can't confirm a refresh would help; that requires an actual experiment.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['cheat'] = df['opportunity_flags'] * 0.99
print("Cheat Correlation:", df['cheat'].corr(df['opportunity_flags']))
df.drop(columns=['cheat'], inplace=True)
print("cheat column removed")

Cheat Correlation: 0.9999999999999972
cheat column removed


## Ranked action queue

Pages sorted by risk-flag count, then by impression volume within each tier, so review time goes to higher-flag, higher-traffic pages first. Confidence reflects how many independent signals agree (0-1 flags = low/medium, 2-3 = high).

In [17]:
def suggest_action(row):
    if row['is_stale'] and row['is_declining']:
        return 'refresh'
    elif row['is_page_one'] and row['is_declining']:
        return 'protect + monitor closely'
    elif row['is_stale']:
        return 'refresh'
    elif row['is_declining']:
        return 'monitor'
    else:
        return 'no action needed'
df['suggested_action'] = df.apply(suggest_action, axis=1)
def confidence(flags):
    if flags >= 2:
        return 'high'
    elif flags == 1:
        return 'medium'
    else:
        return 'low'
df['confidence'] = df['opportunity_flags'].apply(confidence)
ranked_queue = df.sort_values(
    by=['opportunity_flags', 'total_impressions'],
    ascending=[False, False]).reset_index(drop=True)
top_20 = ranked_queue[[
    'content_hash_id', 'opportunity_flags', 'reason_code',
    'suggested_action', 'confidence', 'total_impressions']].head(20)
print("Top 20 pages for review:")
print(top_20.to_string(index=False))

Top 20 pages for review:
         content_hash_id  opportunity_flags                                           reason_code          suggested_action confidence  total_impressions
content_42ce26be1ec6be00                  3 declining traffic, page-one decay risk, stale content                   refresh       high               4411
content_fd994a655f0d049b                  3 declining traffic, page-one decay risk, stale content                   refresh       high                290
content_246e3d9094199845                  3 declining traffic, page-one decay risk, stale content                   refresh       high                 31
content_ec2e0346994fb5a5                  2                declining traffic, page-one decay risk protect + monitor closely       high             245276
content_7172a7fad43f0998                  2                declining traffic, page-one decay risk protect + monitor closely       high             205867
content_8d7d99f109e19aa2                  2        

## Self-check

Before you submit, confirm each line honestly:

- [ Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes ] No client names, URLs, or private queries anywhere
- [ Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.